In [1]:
import os, sys

if not os.path.isdir("Explanations"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("cwd:", os.getcwd())
print("Explanations/ exists:", os.path.isdir("Explanations"))


cwd: c:\Users\Offic\OneDrive\Desktop\Projects\5 - XAI Evaluation Paper
Explanations/ exists: True


In [2]:
from src.evaluation import create_experiment
from src.robustness import run_robustness
from src.utils import Logger
import os

os.makedirs("Evaluation/logs", exist_ok=True)
# logger = Logger("Evaluation/logs", filename="setup_log.txt")
# experiment_id = create_experiment("Evaluation", {"purpose": "fidelity run"

DATASETS = [
    ("healthcare", "pima_diabetes"),
    ("healthcare", "breast_cancer_wisconsin"),
    ("healthcare", "heart_disease_uci"),
    ("finance", "loan_default"),
    ("finance", "financial_distress"),
    ("finance", "credit_card_fraud_2023"),
]
MODEL_NAMES = ["rf", "xgb"]

logger = Logger("Evaluation/logs", filename="ledger.log")
experiment_id = create_experiment(
    "Evaluation",
    {"phase": "robustness", "datasets": DATASETS, "models": MODEL_NAMES},
    logger,
)
print("experiment_id:", experiment_id)

Created experiment_id=exp_20260718T135811_33e8ebcb, logged to Evaluation\run_ledger.csv
experiment_id: exp_20260718T135811_33e8ebcb


In [ ]:
# run_robustness({
#     "domain": domain, "dataset_name": dataset_name, "model_name": model_name,
#     "experiment_id": experiment_id,
# })

In [ ]:
import os
from src.robustness import run_robustness

results = {}
for domain, dataset_name in DATASETS:
    for model_name in MODEL_NAMES:
        run_dir = os.path.join("Explanations", domain, dataset_name, model_name)
        if not os.path.isdir(run_dir):
            print(f"SKIP {domain}/{dataset_name}/{model_name}: no Explanations/ artifacts yet")
            continue
        try:
            rows = run_robustness({
                "domain": domain,
                "dataset_name": dataset_name,
                "model_name": model_name,
                "experiment_id": experiment_id,
                # "robustness_root": "somewhere/else",
            })
            results[(domain, dataset_name, model_name)] = len(rows)
        except Exception as e:
            print(f"FAILED {domain}/{dataset_name}/{model_name}: {e}")

print()
print(f"{len(results)}/{len(DATASETS) * len(MODEL_NAMES)} combos completed.")
results


------------------------------------------------------------
ROBUSTNESS: pima_diabetes x rf
------------------------------------------------------------
Loaded model (skops): Models\healthcare\pima_diabetes\rf\model\model.skops
Feature-group map for pima_diabetes: 8 original features (from 8 model-input columns).
Computed std for 8 numerical feature(s) from Datasets\healthcare\pima_diabetes\processed\data\X_train_prebalance.csv.
shap: robustness computed for 154 instance(s).
lime: robustness computed for 154 instance(s).
Appended 616 row(s) to Evaluation\metrics_long.csv

------------------------------------------------------------
DONE
------------------------------------------------------------
Robustness: 616 metrics_long rows written across ['shap', 'lime'].

------------------------------------------------------------
ROBUSTNESS: pima_diabetes x xgb
------------------------------------------------------------
Loaded model (skops): Models\healthcare\pima_diabetes\xgb\model\model.s

{('healthcare', 'pima_diabetes', 'rf'): 616,
 ('healthcare', 'pima_diabetes', 'xgb'): 616,
 ('healthcare', 'breast_cancer_wisconsin', 'rf'): 456,
 ('healthcare', 'breast_cancer_wisconsin', 'xgb'): 456,
 ('healthcare', 'heart_disease_uci', 'rf'): 736,
 ('healthcare', 'heart_disease_uci', 'xgb'): 736,
 ('finance', 'loan_default', 'rf'): 2000,
 ('finance', 'loan_default', 'xgb'): 2000,
 ('finance', 'financial_distress', 'rf'): 2936,
 ('finance', 'financial_distress', 'xgb'): 2936,
 ('finance', 'credit_card_fraud_2023', 'rf'): 2000,
 ('finance', 'credit_card_fraud_2023', 'xgb'): 2000}

In [4]:
import os

path = "Models/healthcare/pima_diabetes/rf/model/model.skops"
print("exists:", os.path.exists(path))
print("size (bytes):", os.path.getsize(path))

with open(path, "rb") as f:
    header = f.read(50)
print("first 50 bytes:", header)

exists: True
size (bytes): 132
first 50 bytes: b'version https://git-lfs.github.com/spec/v1\noid sha'


In [6]:
import pandas as pd
df = pd.read_csv("Evaluation/metrics_long.csv")

rob_view = df[(df.experiment_id == experiment_id) & (df.metric_property == "Robustness")]
rob_view.groupby(["dataset", "model", "explainer", "metric_name"])["value"].mean().unstack("metric_name")

C:\Users\Offic\AppData\Local\Temp\ipykernel_1996\3431092387.py:2: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Evaluation/metrics_long.csv")


metric_name                              spearman_rank_correlation  \
dataset                 model explainer                              
breast_cancer_wisconsin rf    lime                        0.963652   
                              shap                        0.967129   
                        xgb   lime                        0.961111   
                              shap                        0.978787   
credit_card_fraud_2023  rf    lime                        0.927644   
                              shap                        0.951124   
                        xgb   lime                        0.925283   
                              shap                        0.925303   
financial_distress      rf    lime                        0.751898   
                              shap                        0.861607   
                        xgb   lime                        0.688541   
                              shap                        0.786954   
heart_disease_uci       rf    lime                        0.978410   
                              shap                        0.989847   
                        xgb   lime                        0.976380   
                              shap                        0.940665   
loan_default            rf    lime                        0.983606   
                              shap                        0.876829   
                        xgb   lime                        0.912865   
                              shap                        0.828300   
pima_diabetes           rf    lime                        0.933983   
                              shap                        0.954236   
                        xgb   lime                        0.929344   
                              shap                        0.930581   

metric_name                              top_4_jaccard_overlap  \
dataset                 model explainer                          
breast_cancer_wisconsin rf    lime                         NaN   
                              shap                         NaN   
                        xgb   lime                         NaN   
                              shap                         NaN   
credit_card_fraud_2023  rf    lime                         NaN   
                              shap                         NaN   
                        xgb   lime                         NaN   
                              shap                         NaN   
financial_distress      rf    lime                         NaN   
                              shap                         NaN   
                        xgb   lime                         NaN   
                              shap                         NaN   
heart_disease_uci       rf    lime                         NaN   
                              shap                         NaN   
                        xgb   lime                         NaN   
                              shap                         NaN   
loan_default            rf    lime                         NaN   
                              shap                         NaN   
                        xgb   lime                         NaN   
                              shap                         NaN   
pima_diabetes           rf    lime                    0.907359   
                              shap                    0.914286   
                        xgb   lime                    0.905628   
                              shap                    0.909091   

metric_name                              top_5_jaccard_overlap  
dataset                 model explainer                         
breast_cancer_wisconsin rf    lime                    0.933584  
                              shap                    0.892231  
                        xgb   lime                    0.899332  
                              shap                    0.898496  
credit_card_fraud_2023  rf    lime                    0.919524  
       

In [ ]:
import pandas as pd

def feature_volatility(domain, dataset_name, model_name, explainer):
    path = f"Evaluation/Robustness/{domain}/{dataset_name}/{model_name}/{explainer}/perturbation_attributions.csv"
    raw = pd.read_csv(path)
    raw["abs_diff"] = (raw["original_attribution"] - raw["perturbed_attribution"]).abs()
    return raw.groupby("feature")["abs_diff"].mean().sort_values(ascending=False)

feature_volatility("healthcare", "pima_diabetes", "rf", "shap")